# Amostragem de Unidades Monetárias (AUM) – imobilizado da Suzano S.A.
**Dados SINTÉTICOS** – trabalho acadêmico (Auditoria 1 – UnB, 2026.2). Preparador: matrícula 241003769.

Base normativa: NBC TA 530 – Amostragem em auditoria.

Referência bibliográfica consultada para a aplicação do método: GRAMLING, A. A.; RITTENBERG, L. E.; JOHNSTONE, K. M. *Auditoria*: tradução da 7ª edição norte-americana. Porto Alegre: Cengage Learning, 2016. p. 298–302.

**Antes de começar:** tenha em mãos os três arquivos gerados pelo `01_gerar_base_imobilizado_SIM.ipynb`: `base_imobilizado_SIM.csv.gz`, `documentos_suporte_SIM.csv.gz` e `gabarito_erros_SIM.csv`. Rode os blocos em ordem, com Shift + Enter.

## Bloco 1 – Carregar os arquivos
No Colab, este bloco pede o **upload** dos três arquivos gerados pelo `01_gerar_base_imobilizado_SIM.ipynb`: a base, os documentos suporte e o gabarito. Em seguida, lê os três.

In [ ]:
# =====================================================================
# BLOCO 1 – Carregar os arquivos
# Amostragem de Unidades Monetárias (AUM) – imobilizado SINTÉTICO
# Preparador: matrícula 241003769
# =====================================================================
import numpy as np
import pandas as pd
from scipy.stats import chi2

try:                                   # no Colab: pede o upload dos 3 arquivos
    from google.colab import files
    import os
    faltam = [a for a in ['base_imobilizado_SIM.csv.gz', 'documentos_suporte_SIM.csv.gz', 'gabarito_erros_SIM.csv']
              if not os.path.exists(a)]
    if faltam:
        print('Envie os arquivos:', faltam)
        files.upload()
except ImportError:
    pass

LER = dict(sep=';', decimal=',', encoding='utf-8-sig')
base = pd.read_csv('base_imobilizado_SIM.csv.gz', **LER)
docs = pd.read_csv('documentos_suporte_SIM.csv.gz', **LER)
gabarito = pd.read_csv('gabarito_erros_SIM.csv', **LER)
print(f'Base: {len(base):,} bens | documentos: {len(docs):,} | gabarito: {len(gabarito):,} erros')

## Bloco 2 – Parâmetros e tamanho da amostra
Aplica as fórmulas da AUM, conforme apresentadas em Gramling et al. (2016, p. 300–301):

**I = [IT – (IE × FEE)] ÷ FC**

**n = Valor contábil da população ÷ I**

Onde:
- **I** = intervalo de amostragem
- **IT** = incorreção (distorção) tolerável – R\$ 125.289 mil (materialidade para execução, PT 320)
- **IE** = incorreção (distorção) esperada – 10% de IT
- **FEE** = fator de expansão de erro – 1,50 para risco de detecção de 10% (Ilustração 8.7)
- **FC** = fator de confiabilidade – 2,31 para risco de detecção de 10% (Ilustração 8.7)
- **n** = tamanho da amostra (número de sorteios)

A multiplicação IE × FEE é feita antes da subtração. Os bens com saldo negativo ficam fora da população e são examinados à parte (Gramling et al., 2016, p. 302).

In [ ]:
# =====================================================================
# BLOCO 2 – Parâmetros e tamanho da amostra (NBC TA 530; referência: Gramling et al., 2016, p. 298–302)
# =====================================================================
IT = 125_289_000.00      # distorção tolerável = materialidade para execução (PT 320), em R$
IE = 0.10 * IT           # distorção esperada = 10% do tolerável (auditoria inicial, sem histórico)
RD = 0.10                # risco de detecção (PT 200 e PT 315: risco moderado)

# Ilustração 8.7 de Gramling et al. (2016): fatores de confiabilidade (FC) e de expansão de erro (FEE)
TABELA = {0.01: (4.61, 1.90), 0.05: (3.00, 1.60), 0.10: (2.31, 1.50), 0.15: (1.90, 1.40),
          0.20: (1.61, 1.30), 0.25: (1.39, 1.25), 0.30: (1.21, 1.20), 0.50: (0.70, 1.00)}
FC, FEE = TABELA[RD]

# População: bens com valor contábil positivo (saldos negativos são testados à parte – Gramling et al., 2016, p. 302)
pop = base[base['valor_contabil'] > 0].reset_index(drop=True)
negativos = base[base['valor_contabil'] < 0]
VALOR_POP = pop['valor_contabil'].sum()

INTERVALO = (IT - IE * FEE) / FC
N_PLANEJADO = int(np.ceil(VALOR_POP / INTERVALO))

print(f'População: {len(pop):,} bens | R$ {VALOR_POP/1e3:,.0f} mil')
print(f'Saldos negativos, testados à parte: {len(negativos)} bens | R$ {negativos.valor_contabil.sum()/1e3:,.0f} mil')
print(f'FC = {FC} | FEE = {FEE}')
print(f'I = [IT – (IE × FEE)] ÷ FC = [{IT/1e3:,.0f} – ({IE/1e3:,.0f} × {FEE})] ÷ {FC} = R$ {INTERVALO/1e3:,.1f} mil')
print(f'n = Valor contábil da população ÷ I = {VALOR_POP/1e3:,.0f} ÷ {INTERVALO/1e3:,.1f} = {N_PLANEJADO:,} sorteios')

## Bloco 3 – Sorteio sistemático
Como na Ilustração 8.8: soma os valores bem a bem (valor acumulado), sorteia um **início aleatório** entre zero e o intervalo e seleciona o bem em que cai cada ponto: início, início + intervalo, início + 2 × intervalo… Bens maiores que o intervalo são sempre sorteados (estrato superior).

In [ ]:
# =====================================================================
# BLOCO 3 – Seleção sistemática com início aleatório (Ilustração 8.8)
# =====================================================================
SEMENTE_AMOSTRA = 2025
rng = np.random.default_rng(SEMENTE_AMOSTRA)
INICIO = rng.uniform(0, INTERVALO)                       # início aleatório entre 0 e o intervalo

pop['acumulado'] = pop['valor_contabil'].cumsum()         # valor acumulado, bem a bem
pontos = INICIO + INTERVALO * np.arange(N_PLANEJADO)      # posições sorteadas (R$)
pontos = pontos[pontos <= VALOR_POP]
linha = np.searchsorted(pop['acumulado'].values, pontos)  # bem em que cada ponto "cai"

sorteios = pd.Series(linha).value_counts()
amostra = pop.loc[sorteios.index].copy()
amostra['vezes_sorteado'] = sorteios.values
amostra['estrato'] = np.where(amostra['valor_contabil'] >= INTERVALO, 'Superior (≥ intervalo)', 'Inferior')
amostra = amostra.sort_values('acumulado')

print(f'Início aleatório: R$ {INICIO:,.2f}')
print(f'Pontos de seleção: {len(pontos):,} | bens distintos na amostra: {len(amostra):,}')
print(amostra['estrato'].value_counts())
print(f'Valor coberto pela amostra: R$ {amostra.valor_contabil.sum()/1e3:,.0f} mil '
      f'({amostra.valor_contabil.sum()/VALOR_POP:.1%} da população)')

## Bloco 4 – Inspeção dos documentos
Simula o trabalho do auditor: para cada bem sorteado, compara o valor registrado com o **documento suporte**. Sem documento, ou com documento de 2026, o bem é considerado não comprovado (distorção de 100%). A **contaminação** é a distorção dividida pelo valor registrado.

In [ ]:
# =====================================================================
# BLOCO 4 – Inspeção dos documentos suporte dos bens sorteados
# =====================================================================
am = amostra.merge(docs[['id_bem', 'data_documento', 'valor_documento']], on='id_bem', how='left')
am['data_documento'] = pd.to_datetime(am['data_documento'])

# Valor contábil correto = valor do documento – depreciação; sem documento ou documento de 2026 = bem não comprovado
sem_doc = am['valor_documento'].isna()
doc_2026 = am['data_documento'] > pd.Timestamp('2025-12-31')
am['valor_correto'] = np.where(sem_doc | doc_2026, 0.0, am['valor_documento'] - am['depreciacao_acumulada'])
am['distorcao'] = (am['valor_contabil'] - am['valor_correto']).round(2)
am.loc[am['distorcao'].abs() < 0.01, 'distorcao'] = 0.0
am['contaminacao'] = (am['distorcao'] / am['valor_contabil']).clip(upper=1.0)   # "tainting"
am['achado'] = np.select([sem_doc, doc_2026, am['distorcao'] > 0], ['Sem documento suporte', 'Documento posterior a 31/12/2025', 'Valor acima do documento'], '')

excecoes = am[am['distorcao'] > 0].copy()
print(f'Bens com distorção: {len(excecoes)} de {len(am):,}')
print(excecoes.groupby(['estrato', 'achado']).agg(qtd=('id_bem', 'count'), distorcao_R_mil=('distorcao', lambda s: round(s.sum()/1e3, 1))))

# Saldos negativos: exame de 100% (não entram na AUM)
print(f'\nSaldos negativos examinados (100%): {len(negativos)} bens com depreciação acumulada maior que o custo')

## Bloco 5 – Avaliação da amostra
Calcula o **limite superior de distorção (LSD)** = precisão básica + distorção mais provável + ajuste incremental (conforme Gramling et al., 2016, p. 302). No estrato superior, a distorção entra pelo valor real; no inferior, é projetada (contaminação × intervalo). Se o LSD passar da distorção tolerável, o saldo não pode ser aceito.

In [ ]:
# =====================================================================
# BLOCO 5 – Avaliação: limite superior de distorção (LSD)
# =====================================================================
def fator_limite(k, risco=RD):
    """Fator de limite superior (distribuição de Poisson) para k erros – origem da Ilustração 8.7."""
    return chi2.ppf(1 - risco, 2 * (k + 1)) / 2

sup = excecoes[excecoes['estrato'].str.startswith('Superior')]
inf = excecoes[excecoes['estrato'] == 'Inferior'].sort_values('contaminacao', ascending=False).reset_index(drop=True)

precisao_basica = FC * INTERVALO
distorcao_superior = sup['distorcao'].sum()                       # estrato superior: valor real, sem projeção
inf['projetada'] = inf['contaminacao'] * INTERVALO                # estrato inferior: contaminação × intervalo
inf['fator_incremental'] = [fator_limite(k + 1) - fator_limite(k) - 1 for k in range(len(inf))]
inf['ajuste_incremental'] = inf['contaminacao'] * INTERVALO * inf['fator_incremental']

projetada = inf['projetada'].sum()
incremental = inf['ajuste_incremental'].sum()
dist_mais_provavel = distorcao_superior + projetada
LSD = precisao_basica + dist_mais_provavel + incremental

aval = pd.DataFrame({'Componente': ['Precisão básica (FC × intervalo)', 'Distorção do estrato superior (valor real)',
                                    'Distorção projetada do estrato inferior', 'Distorção mais provável (soma das duas acima)',
                                    'Ajuste incremental', 'LIMITE SUPERIOR DE DISTORÇÃO (LSD)', 'Distorção tolerável (PT 320)'],
                     'R$ mil': [precisao_basica, distorcao_superior, projetada, dist_mais_provavel, incremental, LSD, IT]})
aval['R$ mil'] = (aval['R$ mil'] / 1e3).round(1)
print(aval.to_string(index=False))
print('\nCONCLUSÃO:', 'LSD MAIOR que o tolerável – não se pode concluir que o saldo está livre de distorção relevante.'
      if LSD > IT else 'LSD menor que o tolerável – saldo aceito ao risco de 10%.')

## Bloco 6 – Comparação com o gabarito
Compara os erros **inseridos** (gabarito) com os **encontrados** pela amostra, por tipo de erro.

In [ ]:
# =====================================================================
# BLOCO 6 – Comparação com o gabarito (erros inseridos x encontrados)
# =====================================================================
g = gabarito.copy()
g['na_populacao'] = g['id_bem'].isin(pop['id_bem'])
g['na_amostra'] = g['id_bem'].isin(am['id_bem'])
g['detectado'] = g['id_bem'].isin(excecoes['id_bem'])
comp = g.groupby('codigo_erro').agg(inseridos=('id_bem', 'count'), na_amostra=('na_amostra', 'sum'),
                                    detectados_AUM=('detectado', 'sum'),
                                    distorcao_inserida_R_mil=('distorcao_reais', lambda s: round(s.sum()/1e3, 1)))
comp = comp.loc[sorted(comp.index, key=lambda c: int(c[1:]))]
print(comp)
super_aval = g.loc[g['codigo_erro'].isin(['E1', 'E5', 'E9', 'E10']), 'distorcao_reais'].sum()
print(f'\nSuperavaliação real inserida (E1, E5, E9, E10): R$ {super_aval/1e3:,.1f} mil')
print(f'Distorção mais provável estimada pela AUM:     R$ {dist_mais_provavel/1e3:,.1f} mil')
print(f'Limite superior de distorção (LSD):           R$ {LSD/1e3:,.1f} mil')
print('E2, E3, E4, E6 (em parte), E7 e E8 não são alvo da AUM: são encontrados pelos testes de regras e de lançamentos (PT 240 e PT 500).')

## Bloco 7 – Planilha e download
Grava a planilha `PT_530_Amostra_AUM_SIM.xlsx` com parâmetros, amostra, distorções, avaliação, comparação com o gabarito e saldos negativos, e faz o download.

In [ ]:
# =====================================================================
# BLOCO 7 – Gravar a planilha da amostra e baixar
# =====================================================================
param = pd.DataFrame({'Parâmetro': ['Distorção tolerável (R$)', 'Distorção esperada (R$)', 'Risco de detecção', 'Fator de confiabilidade',
                                    'Fator de expansão de erro', 'Valor da população (R$)', 'Bens na população', 'Intervalo (R$)',
                                    'Sorteios planejados', 'Início aleatório (R$)', 'Semente do sorteio', 'Bens distintos na amostra'],
                      'Valor': [IT, IE, RD, FC, FEE, VALOR_POP, len(pop), INTERVALO, N_PLANEJADO, INICIO, SEMENTE_AMOSTRA, len(am)]})
cols = ['id_bem', 'classe', 'subclasse', 'unidade', 'valor_contabil', 'acumulado', 'vezes_sorteado', 'estrato',
        'valor_documento', 'data_documento', 'valor_correto', 'distorcao', 'contaminacao', 'achado']
with pd.ExcelWriter('PT_530_Amostra_AUM_SIM.xlsx') as w:
    pd.DataFrame({'Aviso': ['DADOS SINTÉTICOS – trabalho acadêmico UnB – preparador 241003769 – não representam registros da Suzano S.A.']}).to_excel(w, sheet_name='Leia-me', index=False)
    param.to_excel(w, sheet_name='Parâmetros', index=False)
    am[cols].to_excel(w, sheet_name='Amostra', index=False)
    excecoes[cols].to_excel(w, sheet_name='Distorções', index=False)
    inf[['id_bem', 'contaminacao', 'projetada', 'fator_incremental', 'ajuste_incremental']].to_excel(w, sheet_name='Projeção inferior', index=False)
    aval.to_excel(w, sheet_name='Avaliação', index=False)
    comp.reset_index().to_excel(w, sheet_name='Gabarito x achados', index=False)
    negativos[['id_bem', 'classe', 'custo_aquisicao', 'depreciacao_acumulada', 'valor_contabil']].to_excel(w, sheet_name='Saldos negativos', index=False)
print('Planilha PT_530_Amostra_AUM_SIM.xlsx gravada.')
try:
    files.download('PT_530_Amostra_AUM_SIM.xlsx')
except Exception:
    pass